# CLM losses — notebook pre-envío

**rev. P1 · 2026-10-11** · código clm_losses 0.3.8 (hash `a466295d82df1b00`)

**Qué es.** Lo que falta correr antes de enviar el paper a JAIR. Está **fuera del protocolo**: no toca el freeze, no re-entrena nada de F3 y no cambia ningún veredicto confirmatorio. Cada bloque guarda en Drive su regla de lectura, con fecha, **antes** de correr, y no la reescribe.

**Cómo.** Con la GPU de siempre (RTX PRO 6000): Entorno de ejecución → **Ejecutar todo**. No hay nada que elegir.

- Tarda unas **3–5 horas**. Lo más lento es el juez (~1–1.5 h para ~25,000 ítems) y las descargas: unos 150 GB entre el juez, los embeddings de entrenamiento y Qwen3-Embedding-8B. Los pesos del juez y de Qwen3-Embedding se borran del disco local al terminar.
- Si la sesión se corta, vuelve a darle **Ejecutar todo**: cada paso retoma donde quedó y no repite lo terminado.
- Si un paso falla, queda anotado y el notebook sigue con el siguiente. El resumen final dice qué pasó en cada uno.

| Bloque | Qué hace | ¿Entrena? | Descarga | Tiempo aprox. |
|---|---|---|---|---|
| 0 | Extrae de Drive lo que ya existe: entorno por fase, F3 frente a F1b, Friedman sin T-gem barajado, puntuaciones secundarias y B4 | no | nada | minutos |
| 1 | Calidad de los sets del sellado: falsos negativos (juez OLMo 3.1 32B sobre todos los ítems de T-gem, T-mine y los 5 generadores del macro) y fuga entre splits; con eso, sensibilidad de C1, C2 y C2c | no | tokens 10.5 GB y juez 64 GB | ~2 h |
| 2 | Condición de borde de B5: 6 denominadores de InfoNCE+HN × 3 semillas, en el split de test (exploratorio) | sí, 18 corridas de ~1 min | embeddings 66 GB | ~1 h |
| 3 | Referencias sin entrenar: Qwen3-Embedding-8B, BM25 y el coseno crudo de Qwen3-8B, con estados reales y barajados | no | 16 GB | ~45 min |

**Lo único que haces a mano: etiquetar 200 ítems (~3 h).**

1. Cuando termine la primera corrida, baja de Drive `CLM_losses/colab/results/PRE/b1_calidad/etiquetas/ETIQUETAR_200_items.csv`. Son 200 preguntas con 10 candidatos cada una (2,000 filas). La celda 1.2 lo deja listo antes de que corra el juez.
2. En la columna `es_correcta(0/1)` pon **1** si el candidato también responde bien la pregunta (dice lo mismo que el oro con otras palabras o con otro formato, o es correcto por otra razón) y **0** si no. Juzga contra el oro: no hace falta el documento.
3. No mires lo que dijo el juez antes de terminar: el acuerdo sólo vale si tus etiquetas son ciegas.
4. Súbelo a esa misma carpeta como `ETIQUETADAS.csv` (o `.xlsx`) y vuelve a darle **Ejecutar todo**. Sólo se recalculan el acuerdo y los informes, en minutos.

**Dónde queda todo.** En `CLM_losses/colab/results/PRE/`. `RESUMEN_PREENVIO.md` junta los cuatro informes; las corridas del bloque 2 quedan en `runs/PRE_borde/`. Al terminar, sube a la conversación el notebook ejecutado y `RESUMEN_PREENVIO.md`.

**Juez de falsos negativos: `allenai/Olmo-3.1-32B-Instruct`** (bf16, Apache-2.0). El diseño proponía gpt-oss-120b, pero no corre en esta GPU: sus kernels MXFP4 no cubren sm_120 y des-cuantizado no cabe. OLMo 3.1 32B ya corrió aquí en el panel T-gen. No es de la familia del oro (Qwen3), ni de la de los HN de entrenamiento (Gemini/Gemma), ni uno de los 5 generadores del macro.

## Preparación

In [ ]:
#@title P.1 · Parámetros, Drive y GPU (fijos: no hay nada que cambiar)
import os, shutil, subprocess, sys
PROFILE = "colab"
DRIVE_BASE = "/content/drive/MyDrive/CLM_losses"
CACHE_ROOT = "/content/cache"
USE_DRIVE_CODE = False
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(os.path.join(DRIVE_BASE, PROFILE)):
    raise SystemExit(f"No encuentro {DRIVE_BASE}/{PROFILE} en tu Drive: abre este notebook con la cuenta del estudio.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("Carpeta del estudio:", os.path.join(DRIVE_BASE, PROFILE))
print("GPU (nombre, driver, memoria):", gpu or "ninguna")
if "RTX PRO 6000" not in gpu:
    print("ATENCIÓN: no es la RTX PRO 6000 de F1b y F3. El bloque 2 compara con esas corridas: con otra GPU el chequeo "
          "de reproducción puede dar diferencias numéricas. Sin GPU, el juez, el bloque 2 y Qwen3-Embedding se saltan.")

In [ ]:
#@title P.2 · Dependencias
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore", *pkgs], check=True)
pip("huggingface_hub>=0.24", "pyarrow>=14", "scipy", "pandas", "matplotlib", "openpyxl", "transformers>=5.5",
    "accelerate>=0.33", "sentence-transformers>=5.0")
# transformers se instala aquí, antes de importarlo: actualizarlo a mitad de sesión mezcla versiones en memoria
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN cargado desde los secretos de Colab")
except Exception:
    print("Sin HF_TOKEN (opcional): los modelos de este notebook no tienen acceso restringido")

In [ ]:
#@title P.3 · Instalar el código (v0.3.8, embebido en el notebook)
import base64, hashlib, io, shutil, zipfile
CODE_ZIP_B64 = "..."
CODE_ZIP_SHA256 = "43a05640abfba03a7fd3e70321a87e2f509c5eaf97b473f0ad0b20830c9a7061"
PKG_ROOT = "/content/clm_losses_pkg"
drive_zip = os.path.join(DRIVE_BASE, "code", "clm_losses_pkg.zip")
if USE_DRIVE_CODE and os.path.exists(drive_zip):
    data, origen = open(drive_zip, "rb").read(), drive_zip
else:
    data, origen = base64.b64decode(CODE_ZIP_B64), "embebido"
    assert hashlib.sha256(data).hexdigest() == CODE_ZIP_SHA256, "el código embebido está corrupto"
shutil.rmtree(PKG_ROOT, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(data)).extractall(PKG_ROOT)
open("/content/clm_code.zip", "wb").write(data)       # para rehacer la Sección 1 sola si la sesión se reinicia
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)
os.environ["PYTHONPATH"] = PKG_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
for m in [m for m in list(sys.modules) if m == "clm_losses" or m.startswith("clm_losses.")]:
    del sys.modules[m]
import clm_losses
from clm_losses.utils import code_hash
CODE_HASH = code_hash()
snap = os.path.join(DRIVE_BASE, "code", CODE_HASH)
if not os.path.exists(snap):
    shutil.copytree(os.path.join(PKG_ROOT, "clm_losses"), snap)
print(f"clm_losses {clm_losses.__version__} ({origen}) · hash {CODE_HASH} · copia legible: {snap}")

In [ ]:
#@title P.4 · Configuración del estudio
import warnings
import torch
from IPython.display import Markdown, display
from clm_losses import preenvio as pre
from clm_losses.pipeline import setup
warnings.filterwarnings("ignore", message="To exit")
study = setup(PROFILE, drive_base=DRIVE_BASE, cache_root=CACHE_ROOT)
GPU = torch.cuda.is_available()

def paso(nombre, fn):
    # corre un paso; si falla, lo anota en results/PRE/ESTADO.json y el notebook sigue
    return pre.correr(study, nombre, fn)

def mostrar(*rel):
    p = pre.pdir(study, *rel)
    if os.path.exists(p):
        display(Markdown(open(p, encoding="utf-8").read()))
    else:
        print("Todavía no hay", p)

print(f"Resultados en {pre.pdir(study)} · GPU: {'sí' if GPU else 'no'}")

## Bloque 0 · Extracción de lo que ya está en Drive

No entrena ni descarga nada. Cierra los [VERIFY] del Diseño del estudio: entorno por fase, si F3 repitió los modelos de F1b, Friedman con 7 bloques, puntuaciones secundarias y B4.

In [ ]:
#@title 0 · Bloque 0: extracción de lo que ya está en Drive (sin entrenar ni descargar)
paso("bloque 0: extracción", lambda: pre.bloque0(study))
mostrar("b0_extraccion", "EXTRACCION.md")

## Bloque 1 · Calidad de los sets del sellado

El diseño lo pedía en F0.4 y F0.5 y no se hizo. Primero salen los textos y el CSV ciego para etiquetar; después corre el juez sobre todos los ítems; luego, la fuga entre splits. El informe aplica la regla escrita antes de correr y recalcula C1, C2 y C2c sin los ítems marcados. Sin exclusiones, ese cálculo tiene que reproducir lo confirmatorio: el informe lo comprueba.

In [ ]:
#@title 1.1 · Textos de test y sellado, y corpus de BM25 (tokens de Nemotron-DQA, 10.5 GB)
paso("bloque 1: textos y corpus de BM25", lambda: pre.paso_textos(study))

In [ ]:
#@title 1.2 · CSV para etiquetar a mano (sale ANTES del juez; se exporta una sola vez)
r = paso("bloque 1: CSV para etiquetar a mano", lambda: pre.paso_etiquetas(study))
if r:
    print("Para etiquetar:", r)
    print("Cuando lo tengas, súbelo a esa carpeta como ETIQUETADAS.csv (o .xlsx) y vuelve a Ejecutar todo.")

In [ ]:
#@title 1.3 · Juez de falsos negativos (OLMo 3.1 32B, ~64 GB; reanudable)
if GPU:
    paso("bloque 1: juez de falsos negativos", lambda: pre.paso_juez(study))
else:
    pre.mark(study, "bloque 1: juez de falsos negativos", "omitido", motivo="sin GPU")
    print("Sin GPU: el juez se salta.")

In [ ]:
#@title 1.4 · Fuga entre splits (pares pregunta–oro repetidos o casi repetidos en entrenamiento)
paso("bloque 1: fuga entre splits", lambda: pre.paso_fuga(study))

In [ ]:
#@title 1.5 · Informe de calidad: tasas, acuerdo con las etiquetas a mano y sensibilidad de C1, C2 y C2c
paso("bloque 1: informe de calidad", lambda: pre.paso_calidad(study))
mostrar("b1_calidad", "CALIDAD.md")

## Bloque 2 · Condición de borde de B5 (exploratoria, split de test)

¿Por qué los oros de otros estados quitan la preferencia sin estado que aprende "sólo HN"? Seis denominadores, con los hiperparámetros de F1b y semillas 0–2. Antes de entrenar las condiciones nuevas se comprueba que "sólo HN" y "HN + in-batch" reproducen sus corridas de referencia; si no, se detiene. Regla de lectura: INNOVATION_MAP §2.8.

In [ ]:
#@title 2 · Bloque 2: condición de borde de B5 (18 corridas en el split de test; reanudable)
if GPU:
    paso("bloque 2: condición de borde", lambda: pre.bloque2(study))
else:
    pre.mark(study, "bloque 2: condición de borde", "omitido", motivo="sin GPU")
    print("Sin GPU: el bloque 2 se salta.")
pre._free_gpu()
mostrar("b2_borde", "BORDE.md")

## Bloque 3 · Referencias sin entrenar

Descriptivas, en test y sellado, con estados reales y barajados. El estado es la pregunta sola para Qwen3-Embedding-8B y BM25.

In [ ]:
#@title 3 · Bloque 3: referencias sin entrenar (Qwen3-Embedding-8B, BM25, coseno crudo)
paso("bloque 3: referencias sin entrenar", lambda: pre.bloque3(study, qwen=GPU))
mostrar("b3_referencias", "REFERENCIAS.md")

## Resumen

In [ ]:
#@title R · Resumen (súbelo a la conversación junto con este notebook ejecutado)
import pandas as pd
md = pre.resumen(study)
est = pre.state(study)
display(pd.DataFrame([{"paso": k, "estado": v.get("estado"), "hora": v.get("hora"),
                       "detalle": v.get("motivo") or v.get("error") or (f"{v.get('minutos')} min" if v.get("minutos") is not None else "")}
                      for k, v in est.items()]))
print("Resumen completo:", pre.pdir(study, "RESUMEN_PREENVIO.md"))
if not pre.read_hand_labels(study):
    print("Falta subir ETIQUETADAS.csv (bloque 1): sin eso, el acuerdo del juez con las etiquetas a mano queda pendiente.")